# 03 · Frames & animation

For a 3D cube (e.g. `time × lat × lon`) pyterraplot serializes each slice along a dim into a **frame**.
Three layers:

- `frames(dim)` → list of full payloads (`lons`/`lats` repeated each frame) + `frame` index & `coord_value`.
- `frames_compact(dim)` → grid stored **once**, only `field` repeats. ~60% smaller; what terraplot's `animate()` wants.
- `frames_to_html` / `frames_compact_to_json` → write them out (animation HTML, or static JSON).

In [ ]:
import numpy as np
import xarray as xr
import pyterraplot  # registers the .tp accessor on DataArray and Dataset

def make_field(nlat=73, nlon=144, phase=0.0, name="t2m",
               long_name="2m temperature anomaly", units="K", holes=True):
    """A smooth, globe-shaped synthetic field on a regular lat/lon grid."""
    lats = np.linspace(90, -90, nlat)
    lons = np.linspace(-180, 180, nlon)
    LON, LAT = np.meshgrid(lons, lats)
    data = (
        8 * np.cos(np.radians(LAT)) * np.sin(np.radians(2 * LON) + phase)
        + 5 * np.sin(np.radians(3 * LON)) * np.cos(np.radians(2 * LAT))
        + 3 * np.cos(np.radians(5 * LON)) * np.sin(np.radians(LAT))
    ).astype(np.float32)
    if holes:
        rng = np.random.default_rng(0)
        data[rng.random((nlat, nlon)) < 0.02] = np.nan  # NaN "missing" cells
    return xr.DataArray(
        data, dims=["lat", "lon"], coords={"lat": lats, "lon": lons},
        name=name, attrs={"units": units, "long_name": long_name},
    )

# Build a time cube: 8 lead-time steps with a drifting phase
times = np.arange(1, 9) * 7  # lead days 7,14,...,56
cube = xr.concat(
    [make_field(phase=p, holes=False) for p in np.linspace(0, 2*np.pi, 8, endpoint=False)],
    dim="time",
).assign_coords(time=times)
cube.name = "t2m"
cube.attrs = {"units": "K", "long_name": "S2S 2m temperature anomaly"}
cube

## `frames()` — full format

In [ ]:
import json
frames = cube.tp.frames(dim="time")
print("n frames        :", len(frames))
print("keys per frame  :", list(frames[0]))
print("frame / coord   :", frames[0]["frame"], frames[0]["coord_value"])
full_kb = len(json.dumps(frames)) / 1024
print(f"full JSON size  : {full_kb:.0f} kB")

## `frames_compact()` — grid stored once

In [ ]:
compact = cube.tp.frames_compact(dim="time")
print("top-level keys :", list(compact))
print("per-frame keys :", list(compact["frames"][0]))
print("coord_values   :", [f["coord_value"] for f in compact["frames"]])
compact_kb = len(json.dumps(compact)) / 1024
print(f"compact size   : {compact_kb:.0f} kB  ({100*(1-compact_kb/full_kb):.0f}% smaller)")

## Writing frames to JSON

For static hosting / loading in your own JS `animate()` call.

In [ ]:
from pathlib import Path
p1 = cube.tp.frames_to_json("frames_full.json", dim="time")
p2 = cube.tp.frames_compact_to_json("frames_compact.json", dim="time")
for p in (p1, p2):
    print(p, f"{Path(p).stat().st_size/1024:.0f} kB")

## `frames_to_html()` — animated, self-contained

Play/pause button, frame scrubber, and a frame label, all inline. Data is gzip-compressed float32 (`pack_frames`) so even a 12-month global run stays small. Works as a 3D globe or any 2D projection.

In [ ]:
from IPython.display import IFrame
cube.tp.frames_to_html("anim_globe.html", dim="time",
                       cmap="RdYlBu_r", interval=500, title="S2S animation (globe)")
print("anim_globe.html", f"{Path('anim_globe.html').stat().st_size/1024:.0f} kB")
IFrame("anim_globe.html", width="100%", height=440)

In [ ]:
cube.tp.frames_to_html("anim_map.html", dim="time", projection="naturalEarth",
                       kind="contourf", levels=12, cmap="RdBu_r",
                       coastlines=True, interval=500, title="S2S animation (2D)")
print("anim_map.html", f"{Path('anim_map.html').stat().st_size/1024:.0f} kB")
IFrame("anim_map.html", width="100%", height=440)